<a href="https://colab.research.google.com/github/obrnocsta/desafios-tecnicos/blob/main/1-desafio/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

In [3]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@graceful-wall-513817-f9.iam.gserviceaccount.com


In [4]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1DshIt2ENx4LqhdScI_9JC0KtDfukfFaFAutHND8QRbM"
NOME_TABELA = "atracoes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Festival_Musica | atracoes | https://docs.google.com/spreadsheets/d/1DshIt2ENx4LqhdScI_9JC0KtDfukfFaFAutHND8QRbM


In [5]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["atracao_id", "nome_artista_ficticio", "genero_musical", "edicao_festival", "dia_apresentacao", "palco", "horario_inicio"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


In [21]:
from datetime import date, timedelta
import random

from faker import Faker


# Configuração dos dados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(90)
random.seed(90)


# Use 10 para cumprir o Passo 9.
# Depois, você pode alterar para 100, 1000 ou 10100.
QUANTIDADE = 10


COLUNAS = [
    "atracao_id",
    "nome_artista_ficticio",
    "genero_musical",
    "edicao_festival",
    "dia_apresentacao",
    "palco",
    "horario_inicio",
]


GENEROS = [
    "pop",
    "rock",
    "eletronica",
    "sertanejo",
    "funk",
    "mpb",
    "rap",
    "forró",
    "reggae",
]

PESOS_GENEROS = [
    16,
    14,
    14,
    14,
    12,
    10,
    10,
    5,
    5,
]


PALCOS = [
    "principal",
    "secundario",
    "eletronico",
    "acustico",
]

PESOS_PALCOS = [
    30,
    35,
    20,
    15,
]


TEMAS_90 = [
    "VHS",
    "Neon",
    "Arcade",
    "Laser",
    "Mutante",
    "Alien",
    "Zumbi",
    "Turbo",
    "Cassete",
    "Monstro",
    "Pixel",
    "Apocalipse",
]

TERMOS_MUSICAIS = [
    "Beat",
    "Band",
    "Crew",
    "Wave",
    "Groove",
    "Mix",
    "Sound",
    "Power",
    "Stars",
    "Noise",
]


def nome_artista():
    """Cria um nome fictício com estilo musical dos anos 1990."""
    prefixo = random.choice(
        ["Os", "As", "DJ", "Banda", "Coletivo", ""]
    )

    nome = f"{random.choice(TEMAS_90)} {random.choice(TERMOS_MUSICAIS)}"

    return f"{prefixo} {nome}".strip()


def gerar_edicoes():
    """Gera 48 edições, seis por ano, entre 1990 e 1997."""
    edicoes = []

    for ano in range(1990, 1998):
        for mes in [1, 3, 5, 7, 9, 11]:
            inicio = date(ano, mes, random.randint(1, 20))

            nome = random.choice(
                [
                    "VHS",
                    "Neon",
                    "Arcade",
                    "Trash",
                    "Laser",
                    "Mutante",
                    "Turbo",
                    "Apocalipse",
                ]
            )

            edicoes.append(
                {
                    "nome": f"Festival {nome} {ano}",
                    "inicio": inicio,
                }
            )

    return edicoes


def gerar_horarios():
    """Cria horários de 12:00 a 23:45, de 15 em 15 minutos."""
    return [
        f"{hora:02d}:{minuto:02d}"
        for hora in range(12, 24)
        for minuto in range(0, 60, 15)
    ]


def gerar_atracoes(quantidade):
    """Gera a quantidade solicitada de atrações fictícias."""
    edicoes = gerar_edicoes()
    horarios = gerar_horarios()
    vagas = []

    for edicao in edicoes:
        for dia in range(3):
            data_show = (
                edicao["inicio"] + timedelta(days=dia)
            ).isoformat()

            for palco in PALCOS:
                for horario in random.sample(
                    horarios,
                    len(horarios),
                ):
                    vagas.append(
                        {
                            "edicao_festival": edicao["nome"],
                            "dia_apresentacao": data_show,
                            "palco": palco,
                            "horario_inicio": horario,
                        }
                    )

    if quantidade > len(vagas):
        raise ValueError(
            f"Quantidade máxima disponível: {len(vagas)}"
        )

    random.shuffle(vagas)

    linhas = []

    for identificador, vaga in enumerate(
        vagas[:quantidade],
        start=1,
    ):
        linhas.append(
            {
                "atracao_id": identificador,
                "nome_artista_ficticio": nome_artista(),
                "genero_musical": random.choices(
                    GENEROS,
                    weights=PESOS_GENEROS,
                    k=1,
                )[0],
                "edicao_festival": vaga["edicao_festival"],
                "dia_apresentacao": vaga["dia_apresentacao"],
                "palco": vaga["palco"],
                "horario_inicio": vaga["horario_inicio"],
            }
        )

    return linhas


# Apaga os dados anteriores e preserva uma aba limpa.
ws.clear()


# Escreve o cabeçalho.
ws.update(
    range_name="A1",
    values=[COLUNAS],
)


# Gera 10 registros para o Passo 9.
linhas = gerar_atracoes(QUANTIDADE)


# Organiza os valores conforme a ordem do cabeçalho.
valores = [
    [linha[coluna] for coluna in COLUNAS]
    for linha in linhas
]


# Insere os dados abaixo do cabeçalho.
ws.append_rows(
    valores,
    value_input_option="RAW",
)


print(len(valores), "linhas enviadas")

10 linhas enviadas


In [22]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 11
Cabeçalho confere: True
[['atracao_id', 'nome_artista_ficticio', 'genero_musical', 'edicao_festival', 'dia_apresentacao', 'palco', 'horario_inicio'], ['1', 'DJ Zumbi Mix', 'eletronica', 'Festival Turbo 1995', '1995-05-20', 'acustico', '21:15'], ['2', 'Banda Apocalipse Noise', 'mpb', 'Festival Arcade 1991', '1991-09-19', 'acustico', '23:30']]
